In [1]:
import pandas as pd
import numpy as np
import random
import copy
import math
from datetime import date
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# KINGElvis FileName
file_name="VTA_CA_OB_KINGElvis.xlsx"

if file_name.split('_')[0].isdigit():
    file_first_name=file_name.split('_')[0]+'_'+file_name.split('_')[1]
else:
    file_first_name=file_name.split('_')[0]

# in some Compeletion Report LSNAMECODE is splited in some it is not so have to check that
def edit_ls_code_column(x):
    value=x.split('_')
    if len(value)>3:
        route_value="_".join(value[:-1])
    else:
        route_value="_".join(value)
    return route_value

# for generated file version
version=2
project_name='VTA'
today_date = date.today()
today_date=''.join(str(today_date).split('-'))

ke_df=pd.read_excel(file_name,sheet_name='Elvis_Review')

In [2]:
wkend_overall_df=pd.read_excel('VTA_CA_CR.xlsx',sheet_name='WkEND-Overall')
# wkend_overall_df['LS_NAME_CODE']=wkend_overall_df['LS_NAME_CODE'].apply(edit_ls_code_column)
wkend_route_df=pd.read_excel('VTA_CA_CR.xlsx',sheet_name='WkEND-RouteTotal')
df=pd.read_csv("reviewtool_20241223_VTA_ROUTE_DIRECTION_CHECk.csv")

# if we have generated route_direction_database file using route_direction_refator_database.py file then have to replace and rename the columns
df.drop(columns=['ROUTE_SURVEYEDCode','ROUTE_SURVEYED'],inplace=True)
df.rename(columns={'ROUTE_SURVEYEDCode_New':'ROUTE_SURVEYEDCode','ROUTE_SURVEYED_NEW':'ROUTE_SURVEYED'},inplace=True) 

wkday_overall_df=pd.read_excel('VTA_CA_CR.xlsx',sheet_name='WkDAY-Overall')
# wkday_overall_df['LS_NAME_CODE']=wkday_overall_df['LS_NAME_CODE'].apply(edit_ls_code_column)
wkday_route_df=pd.read_excel('VTA_CA_CR.xlsx',sheet_name='WkDAY-RouteTotal')


df['ROUTE_SURVEYEDCode_Splited']=df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(str(x).split('_')[:-1]) )


ke_df=ke_df[ke_df['INTERV_INIT']!='999']
ke_df=ke_df[ke_df['INTERV_INIT']!=999]
ke_df=ke_df[ke_df['1st Cleaner']!='No 5 MIN']
ke_df=ke_df[ke_df['1st Cleaner']!='Test']
ke_df=ke_df[ke_df['1st Cleaner']!='Test/No 5 MIN']
# ke_df=ke_df[ke_df['Final_Usage'].str.lower()=='use']



# Getting Data from Database where the Final Usage is Use in KINGELVIS  
df=pd.merge(df,ke_df['id'],on='id',how='inner')
df.drop_duplicates(subset='id',inplace=True)


def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

In [3]:

date_columns_check=['completed','datestarted']
date_columns=check_all_characters_present(df,date_columns_check)

def determine_date(row):
    if not pd.isnull(row[date_columns[0]]):
        return row[date_columns[0]]
    elif not pd.isnull(row[date_columns[1]]):
        return row[date_columns[1]]
    else:
        return pd.NaT

df['Date'] = df.apply(determine_date, axis=1)

def get_day_name(x):
    formats_to_check = ['%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M']
    
    for format_str in formats_to_check:
        try:
            date_object = datetime.strptime(x, format_str)
            day_name = date_object.strftime('%A')
            return day_name
        except ValueError:
            continue

df['Day']=df['Date'].apply(get_day_name)

weekend_df=df[df['Day'].isin(['Saturday','Sunday'])]

weekday_df=df[~(df['Day'].isin(['Saturday','Sunday']))]

In [4]:
#to get the TIMEON column
time_column_check=['timeoncode']
time_column=check_all_characters_present(df,time_column_check)

# to get ROUTE_SURVEYEDCode column from database File
route_survey_column_check=['routesurveyedcode']
route_survey_column=check_all_characters_present(df,route_survey_column_check)

In [5]:
weekday_df[time_column[0]].unique()

array(['MID3', 'MID7', 'MID5', 'MID4', nan, 'PM4', 'MID6', 'PM1', 'PM2',
       'PM3', 'PM5', 'MID2', 'PM7', 'AM3', 'PM6', 'PM8', 'MID1', 'PM9',
       'AM2', 'AM1'], dtype=object)

In [16]:
weekday_df[weekday_df[time_column[0]]=='PM7'].shape

(318, 827)

In [26]:
weekend_df.shape

(94, 827)

In [31]:
weekend_df[weekend_df[time_column[0]]=='MID7'].shape

(6, 827)

In [5]:
wkend_overall_df.dropna(subset=['LS_NAME_CODE'],inplace=True)
wkday_overall_df.dropna(subset=['LS_NAME_CODE'],inplace=True)


In [6]:
def create_route_direction_level_df(overalldf, df):
    """Create Route SurveyedCode Direction-wise comparison in terms of time values."""
    # Define time value groups
    all_time_values = [
    'AM1', 'AM2', 'AM3', 'AM4', 'MID1', 'MID2', 'MID7', 
    'MID3', 'MID4', 'MID5', 'MID6', 'PM1', 'PM2', 'PM3', 
    'PM4', 'PM5', 'PM6', 'PM7', 'PM8', 'PM9'
    ]
    pre_early_am_column=[0]  #0 is for Pre-Early AM header
    early_am_column=[1]  #1 is for Early AM header
    am_column=[2] #This is for AM header
    midday_colum=[3] #this is for MIDDAY header
    pm_column=[4] #this is for PM header
    evening_column=[5] #this is for EVENING header
    
    new_df=pd.DataFrame()
    for value in all_time_values:
        subset_df = df[(df[time_column[0]]==value)]
        print(subset_df.shape[0])
        
#     unique_routes = overalldf['LS_NAME_CODE'].unique()
#     for route_code in unique_routes:
#         # Initialize counts for each time value group
#         counts = {}
#         for group, time_values in time_value_groups.items():
#             # Filter the DataFrame based on the route code and time values
#             subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code) & 
#                            (df[time_column[0]].isin(time_values))]
#             counts[group] = subset_df['id'].nunique()  # Count unique IDs

#         # Append the counts to the result
#         result.append({"ROUTE_SURVEYEDCode": route_code, **counts})

#     # Convert the result to a DataFrame
#     result_df = pd.DataFrame(result)

#     # Rename columns to match desired headers
#     result_df.rename(columns={
#         "Pre-Early AM": 0,
#         "Early AM": 1,
#         "AM": 2,
#         "Midday": 3,
#         "PM": 4,
#         "Evening": 5
#     }, inplace=True)

#     return result_df

In [8]:
def create_time_value_df(df):
    """Create Route SurveyedCode Direction-wise comparison in terms of time values."""
    # Define time value groups
    pre_early_am_values = ['AM1']
    early_am_values = ['AM2']
    am_values = ['AM3', 'AM4', 'MID1', 'MID2', 'MID7']
    midday_values = ['MID3', 'MID4', 'MID5', 'MID6', 'PM1']
    pm_values = ['PM2', 'PM3', 'PM4', 'PM5']
    evening_values = ['PM6', 'PM7', 'PM8', 'PM9']

    # Mapping time groups to corresponding columns
    time_group_mapping = {
        0: pre_early_am_values,
        1: early_am_values,
        2: am_values,
        3: midday_values,
        4: pm_values,
        5: evening_values,
    }

    # Initialize the new DataFrame
    new_df = pd.DataFrame(columns=["Original Text", 0, 1, 2, 3, 4, 5])

    # Populate the DataFrame with counts
    for col, values in time_group_mapping.items():
        for value in values:
            count = weekday_df[weekday_df[time_column[0]] == value].shape[0]
            row = {"Original Text": value}

            # Initialize all columns to 0
            for c in range(6):
                row[c] = 0

            # Update the corresponding column with the count
            row[col] = count
            new_df = pd.concat([new_df, pd.DataFrame([row])], ignore_index=True)

    return new_df


In [23]:
def create_time_value_df(df):
    """
    Create a time-value DataFrame summarizing counts and time ranges.

    Parameters:
        df (pd.DataFrame): Input DataFrame containing the time values.
        time_column (str): Name of the column in the input DataFrame containing the time values.

    Returns:
        pd.DataFrame: Processed DataFrame with counts, time ranges, and display text.
    """
    # Define time value groups
    pre_early_am_values = ['AM1']
    early_am_values = ['AM2']
    am_values = ['AM3', 'AM4', 'MID1', 'MID2', 'MID7']
    midday_values = ['MID3', 'MID4', 'MID5', 'MID6', 'PM1']
    pm_values = ['PM2', 'PM3', 'PM4', 'PM5']
    evening_values = ['PM6', 'PM7', 'PM8', 'PM9']

    # Mapping time groups to corresponding columns
    time_group_mapping = {
        0: pre_early_am_values,
        1: early_am_values,
        2: am_values,
        3: midday_values,
        4: pm_values,
        5: evening_values,
    }

    # Mapping time values to time ranges
    time_mapping = {
        'AM1': 'Before 5:00 am',
        'AM2': '5:00 am - 6:00 am',
        'AM3': '6:00 am - 7:00 am',
        'MID1': '7:00 am - 8:00 am',
        'MID2': '8:00 am - 9:00 am',
        'MID7': '9:00 am - 10:00 am',
        'MID3': '10:00 am - 11:00 am',
        'MID4': '11:00 am - 12:00 pm',
        'MID5': '12:00 pm - 1:00 pm',
        'MID6': '1:00 pm - 2:00 pm',
        'PM1': '2:00 pm - 3:00 pm',
        'PM2': '3:00 pm - 4:00 pm',
        'PM3': '4:00 pm - 5:00 pm',
        'PM4': '5:00 pm - 6:00 pm',
        'PM5': '6:00 pm - 7:00 pm',
        'PM6': '7:00 pm - 8:00 pm',
        'PM7': '8:00 pm - 9:00 pm',
        'PM8': '9:00 pm - 10:00 pm',
        'PM9': 'After 10:00 pm'
    }

    # Initialize the new DataFrame
    new_df = pd.DataFrame(columns=["Original Text", 0, 1, 2, 3, 4, 5])

    # Populate the DataFrame with counts
    for col, values in time_group_mapping.items():
        for value in values:
            count = df[df[time_column[0]] == value].shape[0]
            row = {"Original Text": value}

            # Initialize all columns to 0
            for c in range(6):
                row[c] = 0

            # Update the corresponding column with the count
            row[col] = count
            new_df = pd.concat([new_df, pd.DataFrame([row])], ignore_index=True)

    # Map time values to time ranges
    new_df['Time Range'] = new_df['Original Text'].map(time_mapping)

    # Drop rows with missing time ranges
    new_df.dropna(subset=['Time Range'], inplace=True)

    # Add a display text column with sequential numbering
    new_df['Display_Text'] = range(1, len(new_df) + 1)

    return new_df


In [24]:
wkend_time_df=create_time_value_df(weekday_df)
# wkday_route_direction_df=create_route_direction_level_df(wkday_overall_df,weekday_df)

In [25]:
wkend_time_df

,Original Text,0,1,2,3,4,5,Time Range,Display_Text
0,AM1,11,0,0,0,0,0,Before 5:00 am,1
1,AM2,0,22,0,0,0,0,5:00 am - 6:00 am,2
2,AM3,0,0,258,0,0,0,6:00 am - 7:00 am,3
4,MID1,0,0,524,0,0,0,7:00 am - 8:00 am,4
5,MID2,0,0,506,0,0,0,8:00 am - 9:00 am,5
6,MID7,0,0,371,0,0,0,9:00 am - 10:00 am,6
7,MID3,0,0,0,534,0,0,10:00 am - 11:00 am,7
8,MID4,0,0,0,371,0,0,11:00 am - 12:00 pm,8
9,MID5,0,0,0,560,0,0,12:00 pm - 1:00 pm,9
10,MID6,0,0,0,410,0,0,1:00 pm - 2:00 pm,10


In [10]:
detail_df=pd.read_excel('details_vta_CA_od_excel.xlsx',sheet_name='TOD')
detail_df.head()

,OPPO_TIME[CODE],TIME_ON[Code],TIME_ON,TIME_PERIOD[Code],TIME_PERIOD,START_TIME,AGENCY,WKEND_TIME_PERIOD[Code],WKEND_TIME_PERIOD,Unnamed: 9,...,Unnamed: 11,Unnamed: 12,BOARD_TIMEPERIOD[Code],BOARD_TIMEPERIOD,TIME_PERIOD[Code]2,TIME_PERIOD2,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,AM1,AM1,Before 5:00 am,0,OWL,01:00:00,NaN,NaN,NaN,NaN,...,NaN,NaN,AM1,12:00 am - 4:59 am,0.0,OWL,NaN,AM1 = 12:00 am - 4:59 am,NaN,0 = OWL
1,AM2,AM2,5:00 am - 6:00 am,1,EARLY AM,05:00:00,NaN,NaN,NaN,NaN,...,NaN,NaN,AM2,5:00 am - 5:59 am,1.0,EARLY AM,NaN,AM2 = 5:00 am - 5:59 am,NaN,1 = EARLY AM
2,AM3,AM3,6:00 am - 7:00 am,2,AM,06:00:00,NaN,NaN,NaN,NaN,...,NaN,NaN,AM3,6:00 am - 9:59 am,2.0,AM,NaN,AM3 = 6:00 am - 9:59 am,NaN,2 = AM
3,MID1,MID1,7:00 am - 8:00 am,2,AM,07:00:00,NaN,NaN,NaN,NaN,...,NaN,NaN,MID,10:00 am - 2:59 pm,3.0,MID,NaN,MID = 10:00 am - 2:59 pm,NaN,3 = MID
4,MID2,MID2,8:00 am - 9:00 am,2,AM,08:00:00,NaN,NaN,NaN,NaN,...,NaN,NaN,PM1,3:00 pm - 6:59 pm,4.0,PM,NaN,PM1 = 3:00 pm - 6:59 pm,NaN,4 = PM


In [11]:
detail_df[['OPPO_TIME[CODE]', 'TIME_ON[Code]', 'TIME_ON', 'TIME_PERIOD[Code]',
       'TIME_PERIOD', 'START_TIME']]

,OPPO_TIME[CODE],TIME_ON[Code],TIME_ON,TIME_PERIOD[Code],TIME_PERIOD,START_TIME
0,AM1,AM1,Before 5:00 am,0,OWL,01:00:00
1,AM2,AM2,5:00 am - 6:00 am,1,EARLY AM,05:00:00
2,AM3,AM3,6:00 am - 7:00 am,2,AM,06:00:00
3,MID1,MID1,7:00 am - 8:00 am,2,AM,07:00:00
4,MID2,MID2,8:00 am - 9:00 am,2,AM,08:00:00
5,MID7,MID7,9:00 am - 10:00 am,2,AM,09:00:00
6,MID3,MID3,10:00 am - 11:00 am,3,MID,10:00:00
7,MID4,MID4,11:00 am - 12:00 pm,3,MID,11:00:00
8,MID5,MID5,12:00 pm - 1:00 pm,3,MID,12:00:00
9,MID6,MID6,1:00 pm - 2:00 pm,3,MID,13:00:00


In [12]:
detail_df['TIME_ON[Code]'].values

array(['AM1', 'AM2', 'AM3', 'MID1', 'MID2', 'MID7', 'MID3', 'MID4',
       'MID5', 'MID6', 'PM1', 'PM2', 'PM3', 'PM4', 'PM5', 'PM6', 'PM7',
       'PM8', 'PM9'], dtype=object)

In [13]:
detail_df[['TIME_ON[Code]','TIME_ON']].values

array([['AM1', 'Before 5:00 am'],
       ['AM2', '5:00 am - 6:00 am'],
       ['AM3', '6:00 am - 7:00 am'],
       ['MID1', '7:00 am - 8:00 am'],
       ['MID2', '8:00 am - 9:00 am'],
       ['MID7', '9:00 am - 10:00 am'],
       ['MID3', '10:00 am - 11:00 am'],
       ['MID4', '11:00 am - 12:00 pm'],
       ['MID5', '12:00 pm - 1:00 pm'],
       ['MID6', '1:00 pm - 2:00 pm'],
       ['PM1', '2:00 pm - 3:00 pm'],
       ['PM2', '3:00 pm - 4:00 pm'],
       ['PM3', '4:00 pm - 5:00 pm'],
       ['PM4', '5:00 pm - 6:00 pm'],
       ['PM5', '6:00 pm - 7:00 pm'],
       ['PM6', '7:00 pm - 8:00 pm'],
       ['PM7', '8:00 pm - 9:00 pm'],
       ['PM8', '9:00 pm - 10:00 pm'],
       ['PM9', 'After 10:00 pm']], dtype=object)

In [41]:
time_mapping = {
    'AM1': 'Before 5:00 am',
    'AM2': '5:00 am - 6:00 am',
    'AM3': '6:00 am - 7:00 am',
    'MID1': '7:00 am - 8:00 am',
    'MID2': '8:00 am - 9:00 am',
    'MID7': '9:00 am - 10:00 am',
    'MID3': '10:00 am - 11:00 am',
    'MID4': '11:00 am - 12:00 pm',
    'MID5': '12:00 pm - 1:00 pm',
    'MID6': '1:00 pm - 2:00 pm',
    'PM1': '2:00 pm - 3:00 pm',
    'PM2': '3:00 pm - 4:00 pm',
    'PM3': '4:00 pm - 5:00 pm',
    'PM4': '5:00 pm - 6:00 pm',
    'PM5': '6:00 pm - 7:00 pm',
    'PM6': '7:00 pm - 8:00 pm',
    'PM7': '8:00 pm - 9:00 pm',
    'PM8': '9:00 pm - 10:00 pm',
    'PM9': 'After 10:00 pm'
}

In [42]:
wkend_time_df['Time Range'] = wkend_time_df['Original Text'].map(time_mapping)

In [43]:
wkend_time_df

,Original Text,0,1,2,3,4,5,Time Range
0,AM1,0,0,0,0,0,0,Before 5:00 am
1,AM2,0,0,0,0,0,0,5:00 am - 6:00 am
2,AM3,0,0,4,0,0,0,6:00 am - 7:00 am
3,AM4,0,0,0,0,0,0,NaN
4,MID1,0,0,0,9,0,0,7:00 am - 8:00 am
5,MID2,0,0,0,5,0,0,8:00 am - 9:00 am
6,MID7,0,0,0,6,0,0,9:00 am - 10:00 am
7,MID3,0,0,0,0,19,0,10:00 am - 11:00 am
8,MID4,0,0,0,0,11,0,11:00 am - 12:00 pm
9,MID5,0,0,0,0,8,0,12:00 pm - 1:00 pm


In [44]:
wkend_time_df.dropna(subset=['Time Range'],inplace=True)

In [48]:
wkend_time_df['Display_Text']=range(1,len(wkend_time_df)+1)

In [45]:
count=0
for _,row  in wkend_time_df.iterrows():
    count+=1
    wkend_time_df.loc[row.name,'Display_Text']=count

In [49]:
wkend_time_df

,Original Text,0,1,2,3,4,5,Time Range,Display_Text,Sort
0,AM1,0,0,0,0,0,0,Before 5:00 am,1.0,1
1,AM2,0,0,0,0,0,0,5:00 am - 6:00 am,2.0,2
2,AM3,0,0,4,0,0,0,6:00 am - 7:00 am,3.0,3
4,MID1,0,0,0,9,0,0,7:00 am - 8:00 am,4.0,4
5,MID2,0,0,0,5,0,0,8:00 am - 9:00 am,5.0,5
6,MID7,0,0,0,6,0,0,9:00 am - 10:00 am,6.0,6
7,MID3,0,0,0,0,19,0,10:00 am - 11:00 am,7.0,7
8,MID4,0,0,0,0,11,0,11:00 am - 12:00 pm,8.0,8
9,MID5,0,0,0,0,8,0,12:00 pm - 1:00 pm,9.0,9
10,MID6,0,0,0,0,10,0,1:00 pm - 2:00 pm,10.0,10


In [ ]:
def create_time_value_df_with_display(df):
    """
    Create a time-value DataFrame summarizing counts and time ranges.

    Parameters:
        df (pd.DataFrame): Input DataFrame containing the time values.
        time_column (str): Name of the column in the input DataFrame containing the time values.

    Returns:
        pd.DataFrame: Processed DataFrame with counts, time ranges, and display text.
    """
    # Define time value groups
    pre_early_am_values = ['AM1']
    early_am_values = ['AM2']
    am_values = ['AM3', 'AM4', 'MID1', 'MID2', 'MID7']
    midday_values = ['MID3', 'MID4', 'MID5', 'MID6', 'PM1']
    pm_values = ['PM2', 'PM3', 'PM4', 'PM5']
    evening_values = ['PM6', 'PM7', 'PM8', 'PM9']

    # Mapping time groups to corresponding columns
    time_group_mapping = {
        0: pre_early_am_values,
        1: early_am_values,
        2: am_values,
        3: midday_values,
        4: pm_values,
        5: evening_values,
    }

    # Mapping time values to time ranges
    time_mapping = {
        'AM1': 'Before 5:00 am',
        'AM2': '5:00 am - 6:00 am',
        'AM3': '6:00 am - 7:00 am',
        'MID1': '7:00 am - 8:00 am',
        'MID2': '8:00 am - 9:00 am',
        'MID7': '9:00 am - 10:00 am',
        'MID3': '10:00 am - 11:00 am',
        'MID4': '11:00 am - 12:00 pm',
        'MID5': '12:00 pm - 1:00 pm',
        'MID6': '1:00 pm - 2:00 pm',
        'PM1': '2:00 pm - 3:00 pm',
        'PM2': '3:00 pm - 4:00 pm',
        'PM3': '4:00 pm - 5:00 pm',
        'PM4': '5:00 pm - 6:00 pm',
        'PM5': '6:00 pm - 7:00 pm',
        'PM6': '7:00 pm - 8:00 pm',
        'PM7': '8:00 pm - 9:00 pm',
        'PM8': '9:00 pm - 10:00 pm',
        'PM9': 'After 10:00 pm'
    }

    # Initialize the new DataFrame
    new_df = pd.DataFrame(columns=["Original Text", 0, 1, 2, 3, 4, 5])

    # Populate the DataFrame with counts
    for col, values in time_group_mapping.items():
        for value in values:
            count = df[df[time_column[0]] == value].shape[0]
            row = {"Original Text": value}

            # Initialize all columns to 0
            for c in range(6):
                row[c] = 0

            # Update the corresponding column with the count
            row[col] = count
            new_df = pd.concat([new_df, pd.DataFrame([row])], ignore_index=True)

    # Map time values to time ranges
    new_df['Time Range'] = new_df['Original Text'].map(time_mapping)

    # Drop rows with missing time ranges
    new_df.dropna(subset=['Time Range'], inplace=True)

    # Add a display text column with sequential numbering
    new_df['Display_Text'] = range(1, len(new_df) + 1)

    return new_df

wkend_time_value_df=create_time_value_df_with_display(weekend_df)